# Premier League Match Prediction: Part 2 - Model Training & Multi-Model Evaluation

In this notebook, we train, validate, and compare several distinct modeling approaches on the feature-engineered Premier League dataset from **Part 1**.

### Models Implemented & Compared:
1. **Baseline Prior**: Predicting match probabilities based strictly on historical league base rates (~46% H, ~26% D, ~28% A).
2. **Regularized Multinomial Logistic Regression**: Linear baseline with `StandardScaler` and `TimeSeriesSplit` cross-validation.
3. **Calibrated XGBoost Classifier**: Non-linear gradient boosted decision trees regularized against sports noise and calibrated with sigmoid scaling.
4. **Dixon-Coles / Bivariate Poisson Goal Model**: Quantitative football analytics standard—predicts attacking/defending parameters and converts expected goals $(\lambda_H, \lambda_A)$ into exact match outcome probabilities.
5. **Ensemble Blend**: Combined Poisson + XGBoost predictions for optimal probability calibration and log-loss.

### Sports Evaluation Metrics:
- **Accuracy**: Proportion of correctly predicted match outcomes.
- **Multi-Class Log Loss (Cross-Entropy)**: Heavy penalty for overconfident incorrect predictions.
- **Ranked Probability Score (RPS)**: Standard football betting metric that recognizes ordinal distance (penalizing predicting a Home Win when an Away Win happens more severely than predicting a Draw).

### Step 1: Imports & Setup
We import scikit-learn, XGBoost, scipy, and data manipulation tools.

In [ ]:
import os
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss
from xgboost import XGBClassifier
from scipy.stats import poisson

print("All modeling packages loaded successfully!")

### Step 2: Load Dataset & Chronological Train/Test Split
Next, we load `dataset/processed_match_dataset.csv`. We drop the first 50 initialization matches to ensure rolling features have stabilized, and perform a strict chronological split (first 80% train, last 20% test).

In [ ]:
data_path = Path('dataset/processed_match_dataset.csv')
df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])

# Trim initial burn-in matches
df = df.iloc[50:].reset_index(drop=True)

feature_cols = [
    'Home_Elo', 'Away_Elo', 'Elo_Diff',
    'Home_Roll_Points_5', 'Away_Roll_Points_5', 'Diff_Form',
    'Home_Roll_GF_5', 'Away_Roll_GF_5', 'Home_Roll_GA_5', 'Away_Roll_GA_5', 'Diff_Goals', 'Diff_GD',
    'Home_Roll_STF_5', 'Away_Roll_STF_5', 'Home_Roll_STA_5', 'Away_Roll_STA_5', 'Diff_ST',
    'Home_Roll_STRatio_5', 'Away_Roll_STRatio_5', 'Diff_STRatio',
    'Home_Venue_Points_5', 'Away_Venue_Points_5', 'Diff_Venue_Strength',
    'Home_Rest_Days', 'Away_Rest_Days', 'Diff_Rest_Days'
]

X = df[feature_cols]
y = df['Target'].values

split_idx = int(len(df) * 0.80)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
df_train, df_test = df.iloc[:split_idx], df.iloc[split_idx:]

print(f"Total matches: {len(df)}")
print(f"Training set: {len(X_train)} matches ({df_train['Date'].min().strftime('%Y-%m-%d')} to {df_train['Date'].max().strftime('%Y-%m-%d')})")
print(f"Testing set:  {len(X_test)} matches ({df_test['Date'].min().strftime('%Y-%m-%d')} to {df_test['Date'].max().strftime('%Y-%m-%d')})")

### Step 3: Evaluation Function (RPS, Log-Loss, Accuracy)
In the next cell, we define the evaluation metrics:
- **Ranked Probability Score (RPS)**: In football, the 3 outcomes are ordered ($H > D > A$). If a model predicts a Home Win, and the result is a Draw, that is less incorrect than if the result is an Away Win. RPS properly accounts for this ordinal distance.

In [ ]:
def calculate_rps(y_true, y_probs):
    """Calculates Ranked Probability Score for 3 outcomes (0: Home, 1: Draw, 2: Away)."""
    n = len(y_true)
    rps_scores = []
    
    for i in range(n):
        obs = np.zeros(3)
        obs[y_true[i]] = 1.0
        
        # Cumulative distributions
        cdf_pred = np.cumsum(y_probs[i])
        cdf_obs = np.cumsum(obs)
        
        # RPS = (1 / (K-1)) * sum((cdf_pred - cdf_obs)^2) for k=1..K-1 (K=3)
        rps = 0.5 * np.sum((cdf_pred[:2] - cdf_obs[:2]) ** 2)
        rps_scores.append(rps)
        
    return np.mean(rps_scores)

def evaluate_predictions(y_true, y_probs, model_name="Model"):
    y_pred = np.argmax(y_probs, axis=1)
    acc = accuracy_score(y_true, y_pred)
    loss = log_loss(y_true, y_probs)
    rps = calculate_rps(y_true, y_probs)
    
    print(f"=== {model_name} Evaluation ===")
    print(f"Accuracy:  {acc * 100:.2f}%")
    print(f"Log Loss:  {loss:.4f}")
    print(f"RPS:       {rps:.4f}")
    return {'Model': model_name, 'Accuracy': acc, 'Log_Loss': loss, 'RPS': rps}

### Step 4: Model 1 - Regularized Logistic Regression (Multinomial)
We standardize feature scales with `StandardScaler` and apply $L_2$ regularization. Time-series cross validation is used to evaluate stability.

In [ ]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(C=0.1, max_iter=1000, random_state=42))
])

# Time-series cross-validation on training data
tscv = TimeSeriesSplit(n_splits=5)
cv_scores = []
for fold, (train_cv, val_cv) in enumerate(tscv.split(X_train), 1):
    lr_pipeline.fit(X_train.iloc[train_cv], y_train[train_cv])
    val_acc = lr_pipeline.score(X_train.iloc[val_cv], y_train[val_cv])
    cv_scores.append(val_acc)
print(f"5-Fold TimeSeriesSplit Mean CV Accuracy: {np.mean(cv_scores) * 100:.2f}%")

# Fit on full training set and evaluate on hold-out test set
lr_pipeline.fit(X_train, y_train)
lr_probs = lr_pipeline.predict_proba(X_test)
lr_metrics = evaluate_predictions(y_test, lr_probs, "Multinomial Logistic Regression")

### Step 5: Model 2 - Calibrated XGBoost Classifier
XGBoost captures non-linear feature interactions (such as when a high-Elo team plays away after a short rest).
To prevent tree overfitting on noisy sports data, we constrain `max_depth=3`, use moderate `subsample=0.8`, and calibrate probabilities with sigmoid scaling.

In [ ]:
xgb_base = XGBClassifier(
    max_depth=3,
    learning_rate=0.03,
    n_estimators=120,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5.0,
    random_state=42,
    eval_metric='mlogloss'
)

# Calibrate probabilities
calibrated_xgb = CalibratedClassifierCV(estimator=xgb_base, method='sigmoid', cv=3)
calibrated_xgb.fit(X_train, y_train)

xgb_probs = calibrated_xgb.predict_proba(X_test)
xgb_metrics = evaluate_predictions(y_test, xgb_probs, "Calibrated XGBoost")

# Fit base model to plot feature importances
xgb_base.fit(X_train, y_train)
importances = pd.Series(xgb_base.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.head(10).plot(kind='barh', color='skyblue')
plt.title('Top 10 Most Predictive Features in XGBoost')
plt.xlabel('Feature Importance (Gain)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Step 6: Model 3 - Bivariate Poisson / Dixon-Coles Goal Model
The Dixon-Coles Poisson model is the gold standard in sports modeling.
Instead of predicting match results directly, it estimates:
1. Each team's attacking potency $\alpha_i$.
2. Each team's defensive vulnerability $\beta_i$.
3. The home pitch advantage $\gamma$.

For any match between Home team $i$ and Away team $j$:
$$\lambda_H = \exp(\mu + \gamma + \alpha_i + \beta_j)$$
$$\lambda_A = \exp(\mu + \alpha_j + \beta_i)$$

We compute the probability grid of all possible scores $(g_H, g_A) \in [0..7]$:
$$P(g_H, g_A) = \text{Poisson}(g_H; \lambda_H) \times \text{Poisson}(g_A; \lambda_A)$$
Summing cells gives exact probabilities for **Home Win**, **Draw**, and **Away Win**, as well as the most likely scorelines.

In [ ]:
class PoissonGoalModel:
    def __init__(self):
        self.attack = {}
        self.defense = {}
        self.home_adv = 0.25
        self.league_avg = 1.35

    def fit(self, matches_df):
        # Compute goal scoring and conceding rates per team
        home_goals = matches_df.groupby('HomeTeam')['FTHG'].agg(['sum', 'count'])
        away_goals = matches_df.groupby('AwayTeam')['FTAG'].agg(['sum', 'count'])
        
        home_conceded = matches_df.groupby('HomeTeam')['FTAG'].agg(['sum', 'count'])
        away_conceded = matches_df.groupby('AwayTeam')['FTHG'].agg(['sum', 'count'])
        
        all_teams = sorted(list(set(matches_df['HomeTeam']).union(set(matches_df['AwayTeam']))))
        total_games = len(matches_df)
        total_home_goals = matches_df['FTHG'].sum()
        total_away_goals = matches_df['FTAG'].sum()
        
        avg_home_scored = total_home_goals / total_games
        avg_away_scored = total_away_goals / total_games
        self.home_adv = np.log(avg_home_scored / avg_away_scored)
        self.league_avg = (total_home_goals + total_away_goals) / (2.0 * total_games)
        
        for t in all_teams:
            scored = (home_goals.loc[t, 'sum'] if t in home_goals.index else 0) + (away_goals.loc[t, 'sum'] if t in away_goals.index else 0)
            games = (home_goals.loc[t, 'count'] if t in home_goals.index else 0) + (away_goals.loc[t, 'count'] if t in away_goals.index else 0)
            conceded = (home_conceded.loc[t, 'sum'] if t in home_conceded.index else 0) + (away_conceded.loc[t, 'sum'] if t in away_conceded.index else 0)
            
            t_avg_scored = (scored / games) if games > 0 else self.league_avg
            t_avg_conceded = (conceded / games) if games > 0 else self.league_avg
            
            self.attack[t] = t_avg_scored / self.league_avg
            self.defense[t] = t_avg_conceded / self.league_avg

    def predict_lambdas(self, home_team, away_team):
        att_h = self.attack.get(home_team, 1.0)
        def_h = self.defense.get(home_team, 1.0)
        att_a = self.attack.get(away_team, 1.0)
        def_a = self.defense.get(away_team, 1.0)
        
        lambda_home = att_h * def_a * self.league_avg * np.exp(self.home_adv / 2.0)
        lambda_away = att_a * def_h * self.league_avg * np.exp(-self.home_adv / 2.0)
        return max(lambda_home, 0.2), max(lambda_away, 0.2)

    def predict_match_probs(self, home_team, away_team, max_goals=8):
        lh, la = self.predict_lambdas(home_team, away_team)
        p_home_goals = [poisson.pmf(i, lh) for i in range(max_goals)]
        p_away_goals = [poisson.pmf(j, la) for j in range(max_goals)]
        
        matrix = np.outer(p_home_goals, p_away_goals)
        p_home_win = np.sum(np.tril(matrix, -1))
        p_draw = np.sum(np.diag(matrix))
        p_away_win = np.sum(np.triu(matrix, 1))
        
        # Normalize
        total = p_home_win + p_draw + p_away_win
        return np.array([p_home_win / total, p_draw / total, p_away_win / total])

# Fit model on training period
poisson_model = PoissonGoalModel()
poisson_model.fit(df_train)

poisson_probs = []
for _, row in df_test.iterrows():
    p = poisson_model.predict_match_probs(row['HomeTeam'], row['AwayTeam'])
    poisson_probs.append(p)
poisson_probs = np.array(poisson_probs)

poisson_metrics = evaluate_predictions(y_test, poisson_probs, "Bivariate Poisson Model")

### Step 7: Model 4 - Optimal Ensemble Blend (Poisson + Calibrated XGBoost)
Now we blend the mathematical domain expectations of the Poisson goal model with the feature-rich predictions of Calibrated XGBoost:
$$P_{\text{Ensemble}} = w \times P_{\text{XGB}} + (1 - w) \times P_{\text{Poisson}}$$

In [ ]:
# Blending weights: 0.60 XGBoost + 0.40 Poisson
ensemble_probs = 0.60 * xgb_probs + 0.40 * poisson_probs
ensemble_metrics = evaluate_predictions(y_test, ensemble_probs, "Poisson + XGBoost Ensemble")

### Step 8: Comprehensive Model Benchmark Comparison Table
We compile all models into a single comparative leaderboard.

In [ ]:
# Historical baseline (prior distribution)
train_priors = np.bincount(y_train) / len(y_train)
prior_probs = np.tile(train_priors, (len(y_test), 1))
baseline_metrics = evaluate_predictions(y_test, prior_probs, "Historical Prior Baseline")

results_df = pd.DataFrame([
    baseline_metrics,
    lr_metrics,
    xgb_metrics,
    poisson_metrics,
    ensemble_metrics
])

results_df['Accuracy'] = results_df['Accuracy'].map(lambda x: f"{x * 100:.2f}%")
results_df['Log_Loss'] = results_df['Log_Loss'].map(lambda x: f"{x:.4f}")
results_df['RPS'] = results_df['RPS'].map(lambda x: f"{x:.4f}")

print("\n===== PREMIER LEAGUE MODEL LEADERBOARD =====")
display(results_df)

### Step 9: Save Trained Models for Interactive Match Predictor
We serialize the trained ensemble, Poisson model, and scaler so Part 3 can perform live predictions.

In [ ]:
models_dir = Path('saved_models')
models_dir.mkdir(exist_ok=True)

with open(models_dir / 'calibrated_xgb.pkl', 'wb') as f:
    pickle.dump(calibrated_xgb, f)

with open(models_dir / 'poisson_model.pkl', 'wb') as f:
    pickle.dump(poisson_model, f)

with open(models_dir / 'feature_cols.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print(f"All model artifacts successfully saved in {models_dir.resolve()}")